# Hafta 6: Naïve Bayes, SVM ve Yapay Sinir Ağları

Bu notebook'ta aşağıdaki konular ele alınmaktadır:
1. Bayes Teoremi – Manuel Hesaplama
2. Veri Seti Yükleme ve Keşif (Heart Disease)
3. Veri Ön İşleme
4. Gaussian Naïve Bayes Modeli
5. Support Vector Machine (SVM) Modeli
6. Yapay Sinir Ağı (MLP) Modeli
7. Model Karşılaştırması ve Değerlendirme
8. Hiperparametre Optimizasyonu
9. Sonuç ve Yorumlar

## 1. Gerekli Kütüphanelerin Yüklenmesi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score,
                             roc_curve, auc, RocCurveDisplay)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 20)
print("Kütüphaneler başarıyla yüklendi!")

## 2. Teorik Arka Plan: Bayes Teoremi

### Bayes Teoremi
$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

### Naïve Bayes Sınıflandırıcı
$$P(C_k | x_1, x_2, ..., x_n) \propto P(C_k) \prod_{i=1}^{n} P(x_i | C_k)$$

**Naïve varsayımı:** Tüm öznitelikler birbirinden **bağımsızdır** (koşullu bağımsızlık).

In [ ]:
# ---------------------------------------------------------------
# Bayes Teoremi - Manuel Hesaplama Örneği
# ---------------------------------------------------------------
# Senaryo: Bir hastalık testi
# P(Hasta)     = 0.01   (prevalans: %1)
# P(+|Hasta)   = 0.95   (duyarlılık/sensitivity)
# P(+|Sağlıklı)= 0.05   (yanlış pozitif oranı)
#
# Soru: Test pozitif çıktığında gerçekten hasta olma olasılığı nedir?
# P(Hasta|+) = ?

P_hasta = 0.01
P_saglikli = 1 - P_hasta
P_pos_hasta = 0.95          # Sensitivity
P_pos_saglikli = 0.05       # False positive rate

# P(+) = P(+|Hasta)*P(Hasta) + P(+|Sağlıklı)*P(Sağlıklı)
P_pos = P_pos_hasta * P_hasta + P_pos_saglikli * P_saglikli

# Bayes Teoremi
P_hasta_pos = (P_pos_hasta * P_hasta) / P_pos

print("=" * 55)
print("Bayes Teoremi - Hastalık Testi Örneği")
print("=" * 55)
print(f"  P(Hasta)            = {P_hasta:.4f}")
print(f"  P(+|Hasta)          = {P_pos_hasta:.4f}")
print(f"  P(+|Sağlıklı)      = {P_pos_saglikli:.4f}")
print(f"  P(+)                = {P_pos:.4f}")
print(f"\n  P(Hasta|+)          = {P_hasta_pos:.4f}")
print(f"\n→ Test pozitif çıksa bile gerçekten hasta olma")
print(f"  olasılığı sadece %{P_hasta_pos*100:.1f}.")
print(f"  Bu, düşük prevalanslı hastalıklarda yaygın bir durumdur.")

## 3. Veri Seti Yükleme ve Keşif

**Heart Disease (Kalp Hastalığı)** veri seti kullanılacaktır.  
Hedef değişken: `target` — 1 = Kalp hastalığı var, 0 = Yok

### Öznitelikler
| Sütun | Açıklama |
|-------|----------|
| age | Yaş |
| sex | Cinsiyet (1=Erkek, 0=Kadın) |
| cp | Göğüs ağrısı tipi (0-3) |
| trestbps | Dinlenme kan basıncı |
| chol | Serum kolesterol (mg/dl) |
| fbs | Açlık kan şekeri > 120 mg/dl (1/0) |
| restecg | Dinlenme EKG sonuçları |
| thalach | Maksimum kalp atış hızı |
| exang | Egzersizle tetiklenen angina (1/0) |
| oldpeak | ST depresyonu |
| slope | ST segment eğimi |
| ca | Floroskopi ile boyanan damar sayısı |
| thal | Talasemi tipi |
| target | Kalp hastalığı (1=Var, 0=Yok) |

In [ ]:
import os

DATA_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                         '..', '..', 'Veri_Setleri_Genel', 'heart_disease.csv')

df = pd.read_csv(DATA_PATH)
print(f"Veri seti boyutu: {df.shape[0]} satır, {df.shape[1]} sütun")
print("\nİlk 5 satır:")
df.head()

In [ ]:
print("Veri tipi bilgisi:")
print(df.dtypes)
print("\nEksik değer sayıları:")
print(df.isnull().sum())
print("\nTemel istatistikler:")
df.describe()

In [ ]:
# Hedef değişken ve öznitelik dağılımları
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sol: Sınıf dağılımı
counts = df['target'].value_counts()
axes[0].bar(['Sağlıklı (0)', 'Hasta (1)'], counts.values,
            color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Hedef Değişken Dağılımı')
axes[0].set_ylabel('Örnek Sayısı')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Orta: Yaş dağılımı
sns.histplot(data=df, x='age', hue='target', kde=True, ax=axes[1],
             palette=['steelblue', 'tomato'])
axes[1].set_title('Yaşa Göre Dağılım')
axes[1].legend(['Sağlıklı', 'Hasta'])

# Sağ: Korelasyon ısı haritası
corr = df.corr()
mask = np.zeros_like(corr)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[2], mask=mask, linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[2].set_title('Korelasyon Matrisi')

plt.tight_layout()
plt.show()

## 4. Veri Ön İşleme ve Eğitim / Test Ayrımı

SVM ve Yapay Sinir Ağları **ölçeklendirmeye duyarlıdır**. Bu nedenle `StandardScaler` kullanılacaktır.  
Naïve Bayes ise ölçeklendirmeden etkilenmez ama tutarlılık için aynı veri üzerinde çalışacağız.

In [ ]:
# Öznitelikler ve hedef değişken
feature_cols = [c for c in df.columns if c != 'target']
X = df[feature_cols].values
y = df['target'].values

# Eğitim / Test ayrımı: %80 eğitim, %20 test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Ölçeklendirme (SVM ve YSA için kritik)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Toplam örnek  : {len(X)}")
print(f"Eğitim seti   : {len(X_train)}  ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test seti     : {len(X_test)}   ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nEğitim – sınıf dağılımı : {np.bincount(y_train)}")
print(f"Test   – sınıf dağılımı : {np.bincount(y_test)}")

---
## 5. Model 1: Gaussian Naïve Bayes

Gaussian NB, sürekli öznitelikler için kullanılır.  
Her sınıf için her özniteliğin **ortalama** (μ) ve **varyans** (σ²) değerlerini öğrenir:

$$P(x_i | C_k) = \frac{1}{\sqrt{2\pi\sigma_k^2}} \exp\left(-\frac{(x_i - \mu_k)^2}{2\sigma_k^2}\right)$$

In [ ]:
# Gaussian Naïve Bayes
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

print("=" * 50)
print("Gaussian Naïve Bayes Sonuçları")
print("=" * 50)
print(f"Eğitim Doğruluğu : {nb_model.score(X_train, y_train):.4f}")
print(f"Test Doğruluğu   : {accuracy_score(y_test, y_pred_nb):.4f}")
print("\nSınıflandırma Raporu:")
print(classification_report(y_test, y_pred_nb,
                            target_names=['Sağlıklı', 'Hasta']))

In [ ]:
# Naïve Bayes: Öğrenilen sınıf parametreleri
print("Naïve Bayes – Sınıf Başına Öğrenilen Ortalamalar (μ):")
print("-" * 60)
means_df = pd.DataFrame(nb_model.theta_, columns=feature_cols,
                        index=['Sağlıklı (0)', 'Hasta (1)'])
print(means_df.round(2).to_string())

print("\nNaïve Bayes – Sınıf Başına Öğrenilen Varyanslar (σ²):")
print("-" * 60)
var_df = pd.DataFrame(nb_model.var_, columns=feature_cols,
                      index=['Sağlıklı (0)', 'Hasta (1)'])
print(var_df.round(2).to_string())

---
## 6. Model 2: Support Vector Machine (SVM)

SVM, iki sınıfı ayıran **en geniş marjinli** hiper düzlemi arar.

### Kernel Fonksiyonları
| Kernel | Formül | Kullanım |
|--------|--------|----------|
| Linear | $K(x,y) = x \cdot y$ | Doğrusal ayrılabilir veri |
| RBF | $K(x,y) = e^{-\gamma\|x-y\|^2}$ | Genel amaçlı (varsayılan) |
| Polynomial | $K(x,y) = (x \cdot y + c)^d$ | Polinom sınırlar |

### C Parametresi
- **Küçük C** → Geniş marjin, daha fazla yanlış sınıflandırmaya tolerans (underfitting riski)
- **Büyük C** → Dar marjin, yanlış sınıflandırmaya az tolerans (overfitting riski)

In [ ]:
# Farklı kernel fonksiyonları ile SVM karşılaştırması
kernels = ['linear', 'rbf', 'poly']
svm_results = {}

print("=" * 55)
print("SVM – Kernel Karşılaştırması")
print("=" * 55)
print(f"{'Kernel':<12} {'Eğitim Doğruluğu':>18} {'Test Doğruluğu':>16}")
print("-" * 48)

for kernel in kernels:
    svm = SVC(kernel=kernel, C=1.0, random_state=42)
    svm.fit(X_train_scaled, y_train)
    train_acc = svm.score(X_train_scaled, y_train)
    test_acc = svm.score(X_test_scaled, y_test)
    svm_results[kernel] = {'model': svm, 'train': train_acc, 'test': test_acc}
    print(f"{kernel:<12} {train_acc:>17.4f} {test_acc:>15.4f}")

# En iyi kernel
best_kernel = max(svm_results, key=lambda k: svm_results[k]['test'])
print(f"\n→ En iyi kernel: {best_kernel} (Test Acc: {svm_results[best_kernel]['test']:.4f})")

In [ ]:
# En iyi kernel ile SVM detaylı sonuçlar
svm_model = svm_results[best_kernel]['model']
y_pred_svm = svm_model.predict(X_test_scaled)

print(f"SVM ({best_kernel}) – Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred_svm,
                            target_names=['Sağlıklı', 'Hasta']))

if best_kernel == 'linear':
    # Linear kernel için öznitelik ağırlıkları
    coefs = np.abs(svm_model.coef_[0])
    indices = np.argsort(coefs)[::-1]
    plt.figure(figsize=(10, 5))
    plt.bar(range(len(feature_cols)), coefs[indices],
            color=plt.cm.viridis(np.linspace(0.2, 0.8, len(feature_cols))),
            edgecolor='black')
    plt.xticks(range(len(feature_cols)),
               [feature_cols[i] for i in indices], rotation=35, ha='right')
    plt.title('SVM (Linear) – Öznitelik Ağırlıkları (|w|)', fontsize=13, fontweight='bold')
    plt.ylabel('|Ağırlık|')
    plt.tight_layout()
    plt.show()

In [ ]:
# SVM: C parametresinin etkisi (RBF kernel)
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
train_accs, test_accs = [], []

for C in C_values:
    svm_c = SVC(kernel='rbf', C=C, random_state=42)
    svm_c.fit(X_train_scaled, y_train)
    train_accs.append(svm_c.score(X_train_scaled, y_train))
    test_accs.append(svm_c.score(X_test_scaled, y_test))

plt.figure(figsize=(10, 5))
plt.plot(range(len(C_values)), train_accs, 'b-o', label='Eğitim Doğruluğu', markersize=6)
plt.plot(range(len(C_values)), test_accs, 'r-s', label='Test Doğruluğu', markersize=6)
plt.xticks(range(len(C_values)), [str(c) for c in C_values])
plt.xlabel('C Parametresi')
plt.ylabel('Doğruluk (Accuracy)')
plt.title('SVM (RBF): C Parametresinin Etkisi', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_c_idx = np.argmax(test_accs)
print(f"En iyi C = {C_values[best_c_idx]} (Test Acc: {test_accs[best_c_idx]:.4f})")

---
## 7. Model 3: Yapay Sinir Ağı (MLP)

### Çok Katmanlı Perceptron (MLP)
- **Giriş katmanı:** 13 nöron (öznitelik sayısı)
- **Gizli katmanlar:** Öğrenme yapan katmanlar
- **Çıkış katmanı:** 1 nöron (ikili sınıflandırma)

### Aktivasyon Fonksiyonları
| Fonksiyon | Formül | Aralık |
|-----------|--------|--------|
| Sigmoid | $\sigma(x) = \frac{1}{1+e^{-x}}$ | (0, 1) |
| ReLU | $f(x) = \max(0, x)$ | [0, ∞) |
| Tanh | $\tanh(x)$ | (-1, 1) |

In [ ]:
# Aktivasyon fonksiyonlarını görselleştirme
x = np.linspace(-5, 5, 200)

sigmoid = 1 / (1 + np.exp(-x))
relu = np.maximum(0, x)
tanh = np.tanh(x)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(x, sigmoid, 'b-', linewidth=2)
axes[0].set_title('Sigmoid', fontweight='bold')
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.3)

axes[1].plot(x, relu, 'r-', linewidth=2)
axes[1].set_title('ReLU', fontweight='bold')
axes[1].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

axes[2].plot(x, tanh, 'g-', linewidth=2)
axes[2].set_title('Tanh', fontweight='bold')
axes[2].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[2].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Aktivasyon Fonksiyonları', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Farklı mimariler ile MLP karşılaştırması
architectures = {
    '(50,)'       : (50,),
    '(100,)'      : (100,),
    '(100, 50)'   : (100, 50),
    '(100, 50, 25)': (100, 50, 25),
}

print("=" * 60)
print("MLP – Mimari Karşılaştırması (activation='relu', max_iter=1000)")
print("=" * 60)
print(f"{'Mimari':<20} {'Eğitim Doğruluğu':>18} {'Test Doğruluğu':>16}")
print("-" * 56)

mlp_results = {}
for name, layers in architectures.items():
    mlp = MLPClassifier(hidden_layer_sizes=layers, activation='relu',
                        max_iter=1000, random_state=42)
    mlp.fit(X_train_scaled, y_train)
    train_acc = mlp.score(X_train_scaled, y_train)
    test_acc = mlp.score(X_test_scaled, y_test)
    mlp_results[name] = {'model': mlp, 'train': train_acc, 'test': test_acc}
    print(f"{name:<20} {train_acc:>17.4f} {test_acc:>15.4f}")

best_arch = max(mlp_results, key=lambda k: mlp_results[k]['test'])
print(f"\n→ En iyi mimari: {best_arch} (Test Acc: {mlp_results[best_arch]['test']:.4f})")

In [ ]:
# En iyi mimari ile MLP detaylı sonuçlar
mlp_model = mlp_results[best_arch]['model']
y_pred_mlp = mlp_model.predict(X_test_scaled)

print(f"MLP {best_arch} – Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred_mlp,
                            target_names=['Sağlıklı', 'Hasta']))

In [ ]:
# Öğrenme eğrisi (Loss Curve)
mlp_loss = MLPClassifier(hidden_layer_sizes=(100, 50), activation='relu',
                         max_iter=500, random_state=42)
mlp_loss.fit(X_train_scaled, y_train)

plt.figure(figsize=(10, 5))
plt.plot(mlp_loss.loss_curve_, 'b-', linewidth=1.5)
plt.xlabel('İterasyon')
plt.ylabel('Loss')
plt.title('MLP Öğrenme Eğrisi (Loss Curve)', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Toplam iterasyon : {mlp_loss.n_iter_}")
print(f"Son loss değeri  : {mlp_loss.loss_curve_[-1]:.6f}")

---
## 8. Model Karşılaştırması ve Değerlendirme

In [ ]:
# Tüm modellerin confusion matrix'leri yan yana
models_dict = {
    'Naïve Bayes': (nb_model, X_test, y_pred_nb),
    f'SVM ({best_kernel})': (svm_model, X_test_scaled, y_pred_svm),
    f'MLP {best_arch}': (mlp_model, X_test_scaled, y_pred_mlp),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, (model, X_data, y_pred)) in zip(axes, models_dict.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Sağlıklı', 'Hasta'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold')

plt.suptitle('Karmaşıklık Matrisleri', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Metrik karşılaştırma tablosu
results = []
predictions = {
    'Naïve Bayes': y_pred_nb,
    f'SVM ({best_kernel})': y_pred_svm,
    f'MLP {best_arch}': y_pred_mlp,
}

for name, y_pred in predictions.items():
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
    })

results_df = pd.DataFrame(results).set_index('Model')
print("Model Karşılaştırma Tablosu:")
print("=" * 65)
print(results_df.round(4).to_string())

In [ ]:
# Metrik karşılaştırma görselleştirmesi
fig, ax = plt.subplots(figsize=(12, 6))

x_pos = np.arange(len(results_df.index))
width = 0.2
colors = ['steelblue', 'seagreen', 'tomato', 'darkorange']

for i, metric in enumerate(results_df.columns):
    bars = ax.bar(x_pos + i * width, results_df[metric], width,
                  label=metric, color=colors[i], edgecolor='black', alpha=0.85)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                f'{height:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x_pos + width * 1.5)
ax.set_xticklabels(results_df.index)
ax.set_ylabel('Skor')
ax.set_ylim(0, 1.1)
ax.set_title('Model Performans Karşılaştırması', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Eğrileri karşılaştırması
fig, ax = plt.subplots(figsize=(8, 6))

# NB: predict_proba mevcut
nb_proba = nb_model.predict_proba(X_test)[:, 1]
fpr_nb, tpr_nb, _ = roc_curve(y_test, nb_proba)
auc_nb = auc(fpr_nb, tpr_nb)
ax.plot(fpr_nb, tpr_nb, 'b-', linewidth=2, label=f'Naïve Bayes (AUC={auc_nb:.3f})')

# SVM: decision_function kullan
svm_scores = svm_model.decision_function(X_test_scaled)
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_scores)
auc_svm = auc(fpr_svm, tpr_svm)
ax.plot(fpr_svm, tpr_svm, 'r-', linewidth=2, label=f'SVM (AUC={auc_svm:.3f})')

# MLP: predict_proba mevcut
mlp_proba = mlp_model.predict_proba(X_test_scaled)[:, 1]
fpr_mlp, tpr_mlp, _ = roc_curve(y_test, mlp_proba)
auc_mlp = auc(fpr_mlp, tpr_mlp)
ax.plot(fpr_mlp, tpr_mlp, 'g-', linewidth=2, label=f'MLP (AUC={auc_mlp:.3f})')

# Şans çizgisi
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Rastgele (AUC=0.500)')

ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('True Positive Rate (TPR)')
ax.set_title('ROC Eğrileri Karşılaştırması', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9. Cross-Validation ile Adil Karşılaştırma

In [ ]:
# 5-Katlı Cross-Validation
cv_models = {
    'Naïve Bayes': GaussianNB(),
    'SVM (RBF)': Pipeline([('scaler', StandardScaler()),
                           ('svm', SVC(kernel='rbf', C=1.0, random_state=42))]),
    'SVM (Linear)': Pipeline([('scaler', StandardScaler()),
                              ('svm', SVC(kernel='linear', C=1.0, random_state=42))]),
    'MLP (100,50)': Pipeline([('scaler', StandardScaler()),
                              ('mlp', MLPClassifier(hidden_layer_sizes=(100, 50),
                                                   max_iter=1000, random_state=42))]),
}

print("5-Katlı Çapraz Doğrulama Sonuçları:")
print("=" * 55)
print(f"{'Model':<20} {'Ortalama':>10} {'Std Dev':>10} {'Min':>8} {'Max':>8}")
print("-" * 55)

cv_results = {}
for name, model in cv_models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name:<20} {scores.mean():>9.4f} {scores.std():>10.4f}"
          f" {scores.min():>7.4f} {scores.max():>7.4f}")

# Boxplot
fig, ax = plt.subplots(figsize=(10, 5))
bp_data = [cv_results[name] for name in cv_results]
bp = ax.boxplot(bp_data, labels=list(cv_results.keys()), patch_artist=True)

colors_bp = ['steelblue', 'tomato', 'salmon', 'seagreen']
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Accuracy')
ax.set_title('5-Katlı CV – Model Karşılaştırması', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## 10. Hiperparametre Optimizasyonu (GridSearchCV)

In [ ]:
# SVM Hiperparametre Optimizasyonu
svm_param_grid = {
    'svm__C': [0.1, 1, 10],
    'svm__kernel': ['linear', 'rbf'],
    'svm__gamma': ['scale', 'auto'],
}

svm_pipeline = Pipeline([('scaler', StandardScaler()),
                         ('svm', SVC(random_state=42))])

svm_grid = GridSearchCV(svm_pipeline, svm_param_grid, cv=5,
                        scoring='accuracy', n_jobs=-1)
svm_grid.fit(X_train, y_train)

print("SVM – GridSearchCV Sonuçları")
print("=" * 50)
print("En iyi parametreler:")
for k, v in svm_grid.best_params_.items():
    print(f"  {k:<20} : {v}")
print(f"\nEn iyi CV Accuracy   : {svm_grid.best_score_:.4f}")
print(f"Test Accuracy        : {svm_grid.score(X_test, y_test):.4f}")

In [ ]:
# MLP Hiperparametre Optimizasyonu
mlp_param_grid = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (100, 50)],
    'mlp__activation': ['relu', 'tanh'],
    'mlp__alpha': [0.0001, 0.001, 0.01],
}

mlp_pipeline = Pipeline([('scaler', StandardScaler()),
                         ('mlp', MLPClassifier(max_iter=1000, random_state=42))])

mlp_grid = GridSearchCV(mlp_pipeline, mlp_param_grid, cv=5,
                        scoring='accuracy', n_jobs=-1)
mlp_grid.fit(X_train, y_train)

print("MLP – GridSearchCV Sonuçları")
print("=" * 50)
print("En iyi parametreler:")
for k, v in mlp_grid.best_params_.items():
    print(f"  {k:<30} : {v}")
print(f"\nEn iyi CV Accuracy   : {mlp_grid.best_score_:.4f}")
print(f"Test Accuracy        : {mlp_grid.score(X_test, y_test):.4f}")

---
## 11. Sonuç ve Yorumlar

### Özet Tablo

| Algoritma | Avantajlar | Dezavantajlar | Uygun Olduğu Durumlar |
|-----------|-----------|---------------|----------------------|
| **Naïve Bayes** | Hızlı, az veri ile çalışır, metin sınıflandırmada etkili | Bağımsızlık varsayımı gerçekçi olmayabilir | Metin sınıflandırma, spam filtreleme, çok sınıflı problemler |
| **SVM** | Yüksek boyutlu veride etkili, kernel trick ile esnek | Büyük veri setlerinde yavaş, parametre ayarı gerekli | Orta büyüklükte veri, net sınıf ayrımı olan problemler |
| **YSA (MLP)** | Karmaşık ilişkileri öğrenebilir, esnek mimari | Yavaş eğitim, kara kutu, çok veri gerektirir | Büyük veri setleri, karmaşık örüntüler |

### Önemli Noktalar
- **Ölçeklendirme:** SVM ve YSA için `StandardScaler` kullanmak kritiktir
- **Hiperparametre ayarı:** `GridSearchCV` ile sistematik arama yapılmalıdır
- **Cross-validation:** Tek bir train/test split yerine CV ile daha güvenilir sonuçlar elde edilir
- **Model seçimi:** Veri setinin boyutu, öznitelik sayısı ve yorumlanabilirlik ihtiyacına göre karar verilmelidir

In [ ]:
# Final: Optimize edilmiş modellerin karşılaştırması
final_models = {
    'Naïve Bayes': nb_model.score(X_test, y_test),
    f'SVM (GridSearch)': svm_grid.score(X_test, y_test),
    f'MLP (GridSearch)': mlp_grid.score(X_test, y_test),
}

print("\n" + "=" * 50)
print("FINAL SONUÇLAR (Test Seti)")
print("=" * 50)
for name, score in final_models.items():
    bar = '█' * int(score * 40)
    print(f"{name:<22} : {score:.4f}  {bar}")

best_model_name = max(final_models, key=final_models.get)
print(f"\n→ En iyi model: {best_model_name} ({final_models[best_model_name]:.4f})")